# Milestone 2 — train Nyaya-3B-v5 on the measured extraction gap

v3 tied with base because it trained on questions that NAMED the section —
something RAG already handles. v5 trains what Eval-v1 showed is actually
missing: with the gold statute already in context, base still missed 71 facts,
41% by citing the wrong section and 21% by dropping the exact number.

**Settings:** GPU **T4 x2**, Internet **On**. Budgeted ~4h, hard-capped.

This is a real experiment. A tie is a valid outcome and would point at model
scale rather than data.

In [ ]:
# --- setup + preflight -------------------------------------------------
# torchao is pinned explicitly: Kaggle ships 0.10.0, and peft's LoRA dispatcher
# calls is_torchao_available(), which RAISES ImportError on anything below
# 0.16.0 rather than returning False. That kills get_peft_model() even though
# this project never quantizes anything.
!pip -q install -U transformers accelerate peft trl datasets rank_bm25 "torchao>=0.16.0"

import os

# ONE GPU ONLY. Kaggle's "T4 x2" makes HF Trainer pick DataParallel, which
# moves inputs to cuda:1 while the model is pinned to cuda:0 -- training died
# at step 0 with "index is on cuda:1, different from other tensors on cuda:0".
# Qwen2.5-3B in fp16 is ~6.2 GB and fits one T4, so the second card buys
# nothing here but a device-placement bug. Set before torch initialises, and
# inherited by every subprocess we spawn.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

import subprocess, sys, time

import torch

def run(cmd, check=True):
    print("$", " ".join(str(c) for c in cmd), flush=True)
    p = subprocess.run([str(c) for c in cmd], text=True)
    if check and p.returncode != 0:
        raise RuntimeError(f"step failed: {' '.join(str(c) for c in cmd)}")
    return p.returncode

# Fail here, not 20 minutes into training, if the peft/torchao conflict remains.
import peft
from peft.import_utils import is_torchao_available
try:
    is_torchao_available()
    print("peft/torchao OK")
except ImportError as exc:
    raise RuntimeError(f"peft cannot build LoRA layers: {exc}") from exc

REPO = "https://github.com/JitendraJha98/nyaya-model.git"
if not os.path.exists("/kaggle/working/nyaya-model"):
    subprocess.run(["git", "clone", "--depth", "1", REPO,
                    "/kaggle/working/nyaya-model"], check=True)
os.chdir("/kaggle/working/nyaya-model")
sys.path.insert(0, "src")

if not torch.cuda.is_available():
    raise RuntimeError("No GPU. Settings -> Accelerator -> GPU T4 x2.")
major, minor = torch.cuda.get_device_capability(0)
arch = f"sm_{major}{minor}"
print(f"GPU: {torch.cuda.get_device_name(0)} ({arch}) | visible devices: "
      f"{torch.cuda.device_count()}")
if arch not in torch.cuda.get_arch_list():
    raise RuntimeError(f"{arch} unsupported by this torch ({torch.cuda.get_arch_list()})")
if torch.cuda.device_count() != 1:
    raise RuntimeError(f"expected exactly 1 visible GPU, saw "
                       f"{torch.cuda.device_count()} — DataParallel will break "
                       f"the single-device model placement")

# Precision must match the hardware. bf16 on Turing is software-emulated and
# several times slower — that cost 4.5h on an eval run. Confirm what we get.
from nyaya.trainer import _native_bf16, pick_dtype
print(f"peft {peft.__version__} | native bf16: {_native_bf16()} "
      f"-> training dtype {pick_dtype()}")
print("preflight OK")

In [ ]:
# --- rebuild the training data (deterministic, gitignored) -------------
run([sys.executable, "scripts/19_generate_extraction_data.py"])
run([sys.executable, "scripts/29_build_grounding_data.py", "--cap-per-act", 400])
run([sys.executable, "scripts/30_build_v5_splits.py"])

import json
rep = json.load(open("reports/v5_dataset_report.json"))
print(json.dumps(rep, indent=1)[:400])
assert rep["eval_leak"] == 0 and rep["sections_straddling"] == 0, "leak check failed"

In [ ]:
# --- SMOKE TRAIN: 20 steps, timed, projected --------------------------
# A slow training run is far more expensive than a slow eval. Prove the loop
# works and measure real step time before committing hours. This guard already
# earned its place: it caught a 14.9h projection (2 epochs x 2,817 examples on
# a T4 at ~152 s/step) that would have consumed the entire remaining quota.
import copy

import yaml

cfg = yaml.safe_load(open("configs/train_v5.yaml"))
eff_batch = (cfg["training"]["per_device_train_batch_size"]
             * cfg["training"]["gradient_accumulation_steps"])
SMOKE_STEPS = 20

smoke = copy.deepcopy(cfg)
smoke["run_name"] = "v5-smoke"
smoke["output_dir"] = "outputs/v5-smoke"
smoke["training"]["num_train_epochs"] = 1
smoke["training"]["save_steps"] = 10_000      # don't write checkpoints
smoke["training"]["eval_steps"] = 10_000
# NB: max_steps is NOT plumbed through training_kwargs, so it would be silently
# ignored. Bound the work by DATA instead: 320 examples / effective batch 16
# = exactly 20 optimizer steps in one epoch.
smoke["data"]["max_examples"] = SMOKE_STEPS * eff_batch
yaml.safe_dump(smoke, open("configs/_v5_smoke.yaml", "w"))

t0 = time.time()
run([sys.executable, "scripts/09_train.py", "configs/_v5_smoke.yaml"])
smoke_s = time.time() - t0

per_step = smoke_s / SMOKE_STEPS
# Project against what the REAL run will actually train on: max_examples caps
# the dataset, so using the full file length here would overstate the run.
n_file = sum(1 for _ in open("data/splits_v5/train.jsonl"))
n_train = min(n_file, cfg["data"].get("max_examples") or n_file)
total_steps = cfg["training"]["num_train_epochs"] * n_train / eff_batch
projected_h = per_step * total_steps / 3600
print(f"\nsmoke: {smoke_s:.0f}s for {SMOKE_STEPS} steps "
      f"({per_step:.1f}s/step, includes model load)")
print(f"training on {n_train}/{n_file} examples x "
      f"{cfg['training']['num_train_epochs']} epoch")
print(f"projected full run: {total_steps:.0f} steps -> ~{projected_h:.1f} h")
if projected_h > 5:
    raise RuntimeError(f"~{projected_h:.1f}h exceeds budget — lower "
                       f"data.max_examples or num_train_epochs.")
print("smoke OK")

In [ ]:
# --- full training run -------------------------------------------------
import glob

t0 = time.time()
run([sys.executable, "scripts/09_train.py", "configs/train_v5.yaml"])
print(f"\ntraining wall clock: {(time.time() - t0)/3600:.2f} h")
print("checkpoints:", sorted(glob.glob("outputs/legal-3b-v5/checkpoint-*")))

## Milestone 3 — paired evaluation vs base

The base predictions are already committed, so only v5 needs generating; the
comparison is then a CPU bootstrap over identical questions.

In [ ]:
# Evaluate the trained adapter on Eval-v1, same retriever as the base run.
ckpts = sorted(glob.glob("outputs/legal-3b-v5/checkpoint-*"),
               key=lambda p: int(p.rsplit("-", 1)[1]))
adapter = ckpts[-1] if ckpts else "outputs/legal-3b-v5"
print("evaluating adapter:", adapter)

run([sys.executable, "scripts/26_eval_v1_run.py", "--adapter", adapter,
     "--dense", "--k", 8, "--split", "all", "--batch-size", 4,
     "--label", "nyaya-3b-v5"])

In [ ]:
# Paired bootstrap vs the committed base predictions. This is the claim test:
# if the 95% CI excludes zero, "better than base" is defensible. If it spans
# zero, it is not — and that gets reported as-is.
import pathlib
import shutil

print("base predictions present:",
      pathlib.Path("outputs/eval-v1/base/predictions.jsonl").exists())
run([sys.executable, "scripts/27_compare_runs.py", "--a", "base", "--b", "nyaya-3b-v5"])

out = pathlib.Path("/kaggle/working/nyaya-v5-results")
out.mkdir(exist_ok=True)
for f in ("reports/eval_v1_results.json", "reports/eval_v1_comparison.json",
          "reports/v5_dataset_report.json"):
    if os.path.exists(f):
        shutil.copy(f, out)
for run_dir in pathlib.Path("outputs/eval-v1").glob("*/predictions.jsonl"):
    shutil.copy(run_dir, out / f"{run_dir.parent.name}_predictions.jsonl")
# The adapter is small (~120MB) and is the actual deliverable — keep it.
if os.path.isdir(adapter):
    shutil.copytree(adapter, out / "adapter", dirs_exist_ok=True)
shutil.make_archive("/kaggle/working/nyaya-v5-results", "zip", out)
print("\nDownload nyaya-v5-results.zip from the Output tab.")